## Calling API from OpneRouter


In [1]:
!pip install openai requests -q

In [2]:
import os
import json
import re
import time
import requests
import subprocess
from openai import OpenAI
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)
FREE_MODEL = "openrouter/free"
TOOL_MODEL = "openrouter/free"

### Your first API call

This is **inference**. You send tokens, get tokens back.  the API just wraps it with HTTP.

In [3]:
response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[{"role": "user", "content": "Hii"}],
    temperature=0.0,
    max_tokens=50,
)

print("Response:", response.choices[0].message.content)
print(f"Tokens — in: {response.usage.prompt_tokens}, out: {response.usage.completion_tokens}")
print(f"Model used: {response.model}")

Response: Hello! How can I help you today? 😊
Tokens — in: 15, out: 11
Model used: google/gemma-4-31b-it-20260402:free


# 🛠️ Part 1: Why Do We Need Tools?

Large Language Models are excellent at reasoning and generating text, but they cannot access real-time information or interact with external systems on their own. To overcome this limitation, we give the model **tools** that it can use whenever additional information or actions are required.

In this module, we'll manually define tools using JSON schemas, let the model decide when to use them, execute the tool ourselves, and return the result back to the model.

### Define tools

We describe tools so the model knows what's available. The model doesn't HAVE these tools — it can only REQUEST them.

In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieve the current weather information for a specified city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city."
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Search Wikipedia and return a short summary about a given topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Topic to search on Wikipedia (e.g., 'Artificial Intelligence')."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

### The model decides to use a tool

In [10]:
r = client.chat.completions.create(
    model=TOOL_MODEL,
    messages=[{"role": "user", "content": "What's the weather in Tokyo?"}],
    tools=tools,
    temperature=0,
)

msg = r.choices[0].message
print(f"Finish reason: {r.choices[0].finish_reason}")
print(f"Content: {msg.content}")
print(f"Tool calls: {msg.tool_calls}")

if msg.tool_calls:
    tc = msg.tool_calls[0]
    print(f"\n→ Model wants to call: {tc.function.name}({tc.function.arguments})")
    print("  It GENERATED JSON requesting a function. Our code must execute it.")

Finish reason: tool_calls
Content: None
Tool calls: [ChatCompletionMessageFunctionToolCall(id='get_weather_dqyfwttd6tcx', function=Function(arguments='{"city": "Tokyo"}', name='get_weather'), type='function', index=0)]

→ Model wants to call: get_weather({"city": "Tokyo"})
  It GENERATED JSON requesting a function. Our code must execute it.


### Complete the tool call loop

`User → Model requests tool → OUR code runs tool → Result back to model → Final answer`

In [12]:
# Fake tool implementations
def get_weather(city):
    fake = {
        "Tokyo": {"temp": "22°C", "condition": "partly cloudy"},
        "London": {"temp": "14°C", "condition": "rainy"},
        "Delhi": {"temp": "38°C", "condition": "sunny"},
    }
    return json.dumps(fake.get(city, {"temp": "unknown", "condition": "unknown"}))

def calculate(expression):
    try:
        allowed = set("0123456789+-*/.() ")
        if not all(c in allowed for c in expression):
            return json.dumps({"error": "Invalid characters"})
        return json.dumps({"result": eval(expression)})
    except Exception as e:
        return json.dumps({"error": str(e)})

TOOL_FNS = {"get_weather": get_weather, "calculate": calculate}

# Execute the tool call
if msg.tool_calls:
    tc = msg.tool_calls[0]
    fn = TOOL_FNS[tc.function.name]
    result = fn(**json.loads(tc.function.arguments))
    print(f"Tool result: {result}")

    # Feed result back to model
    messages = [
        {"role": "user", "content": "What's the weather in Tokyo?"},
        msg,
        {"role": "tool", "tool_call_id": tc.id, "content": result}
    ]

    r2 = client.chat.completions.create(
        model=TOOL_MODEL, messages=messages, tools=tools
    )
    print(f"\nFinal answer: {r2.choices[0].message.content}")
    print("\n→ Full cycle: user → model requests tool → we run it → model answers")

Tool result: {"temp": "22\u00b0C", "condition": "partly cloudy"}

Final answer: The current weather in Tokyo is **22°C** with **partly cloudy** skies. Is there anything else you'd like to know about Tokyo's weather or other details? 😊

→ Full cycle: user → model requests tool → we run it → model answers


# 📝 Module Summary

In this module, we explored one of the most important capabilities of modern AI agents: **Tool Calling**.

Large Language Models are excellent at understanding and generating natural language, but they cannot directly access real-time information, perform external actions, or interact with APIs on their own. To overcome this limitation, we provide the model with a list of available **tools**. Each tool is described using a **JSON schema**, which includes the tool's name, description, input parameters, and required fields. This schema acts as a contract, informing the model about the tools it can use and the information each tool requires.

When a user sends a query, the tool definitions are included in the API request. The model analyzes both the user's prompt and the available tools. If it determines that a tool is required, it **does not execute the tool itself**. Instead, it returns a **tool call** containing the selected function name and the arguments needed to execute it.

Our Python application then receives this tool call, identifies the requested function, executes the corresponding Python code, and obtains the result. The output is then added back to the conversation as a **tool message** and sent to the model in a second API request. Using this additional information, the model generates the final response for the user.

Through this process, we learned that the LLM acts as the **decision maker**, while our application acts as the **executor**. The model decides **which tool to use** and **what arguments to pass**, but the actual execution always happens in our application.

This tool-calling workflow is the foundation of modern AI agents. Whether an agent searches Wikipedia, retrieves weather information, queries a database, performs calculations, or calls any external API, the underlying process remains the same:

**User → LLM → Tool Selection → Tool Execution → Tool Result → LLM → Final Answer**

Understanding this complete workflow prepares us for the next module, where we'll build a **ReAct Agent** that can repeatedly reason, choose tools, observe results, and continue working until it successfully completes a task.